# EA3 — Proyecto final: procesamiento distribuido y cierre del caso

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *35* |
| **Integrantes** | *(Paula Andrea Celis Cano)* |
| **Caso de estudio** | *(Wanderbricks)* |
| **Fecha de entrega** | Domingo 4 de octubre |
| **🎥 Enlace al video** | *(pegar aquí — 8 a 10 minutos, mínimo 3 por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta
> a la propia. Un enlace inaccesible se califica como no entregado.

## Cómo usar esta plantilla

Las celdas en *cursiva* son instrucciones: **reemplácenlas por su contenido.**

| Sección | Qué va ahí | Puntos |
|---|---|---|
| 1 · Contexto | De dónde vienen, en una página | — |
| 2 · Capa oro | La tabla de consumo y el recorrido del dato | 20 |
| 3 · Medición inicial | El plan de ejecución y los tres tiempos | 15 |
| 4 · Optimización | La técnica, el plan nuevo y la comparación | 20 |
| 5 · Límites | Cuándo esa técnica sería contraproducente | 10 |
| 6 · Cierre del caso | La pregunta orientadora y las conclusiones | 10 |
| 7 · Reparto y uso de IA | Quién hizo qué | — |
| ⭐ Opcional | Tablero o Genie sobre la capa oro | +5 |

Los 25 puntos restantes son individuales y se evalúan en el video.

In [0]:
from pyspark.sql import functions as F
import time, statistics

# ⬇️ Cambiar por su catálogo y esquema de la EA1
CATALOGO = "bigdata_grupo35"
ESQUEMA  = "wanderbricks"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")

print(f"Trabajando en {CATALOGO}.{ESQUEMA}\n")
display(spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}"))

# 1 · Contexto

*En un par de párrafos: cuál es su caso, qué construyeron en la EA1 y la EA2, y cuál es la
pregunta de negocio que va a responder la capa oro de este proyecto.*


Wanderbricks es una plataforma de reservas de alojamientos que contiene información
sobre usuarios, propiedades, destinos, reservas, pagos y reseñas.

En esta evidencia se continúa el procesamiento realizado en las capas Bronce y Plata,
llevando los datos hasta la capa Oro. El objetivo es obtener información agregada que
pueda ser utilizada directamente para responder preguntas de negocio.

Para esta evidencia se analiza la información de las reservas, las propiedades y los
destinos. A partir de estos datos se construye una tabla Oro que permite identificar
los destinos que generan mayores ingresos por reservas.

Además, se realiza una medición de una operación distribuida y se aplica una
optimización para comparar su comportamiento antes y después del cambio.

# 2 · Capa oro · 20 puntos

## 2.1 · Qué pregunta responde

### Pregunta de negocio

**¿Cuáles son los destinos que generan mayores ingresos por reservas?**

Esta pregunta puede ser utilizada por el **equipo de negocio o de gestión de Wanderbricks** para identificar los destinos que generan mayores ingresos y conocer la cantidad de reservas y el valor promedio de las reservas.

La tabla Oro presenta información ya agregada y ordenada por ingresos, por lo que una persona del negocio puede consultar directamente los resultados sin tener que realizar transformaciones adicionales.

Las principales métricas son:

- **Destino:** lugar donde se encuentra la propiedad.
- **País:** país del destino.
- **Cantidad de reservas:** número de reservas realizadas.
- **Ingresos totales:** suma de los valores de las reservas.
- **Promedio de reserva:** valor promedio de las reservas.

In [0]:
from pyspark.sql import functions as F

bookings = spark.table("bigdata_grupo35.plata.bookings")
properties = spark.table("samples.wanderbricks.properties")
destinations = spark.table("samples.wanderbricks.destinations")

ingresos_destinos = (
    bookings.alias("b")
    .join(
        properties.alias("p"),
        F.col("b.property_id") == F.col("p.property_id"),
        "inner"
    )
    .join(
        destinations.alias("d"),
        F.col("p.destination_id") == F.col("d.destination_id"),
        "inner"
    )
    .groupBy(
        F.col("d.destination").alias("destino"),
        F.col("d.country").alias("pais")
    )
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(F.sum("b.total_amount"), 2).alias("ingresos_totales"),
        F.round(F.avg("b.total_amount"), 2).alias("promedio_reserva")
    )
    .orderBy(F.desc("ingresos_totales"))
)

print("✅ Consulta de ingresos por destino creada")

In [0]:
ingresos_destinos = spark.table("bigdata_grupo35.oro.ingresos_por_destino")

display(ingresos_destinos)

In [0]:
from pyspark.sql import functions as F

bookings = spark.table("bigdata_grupo35.plata.bookings")
properties = spark.table("samples.wanderbricks.properties")
destinations = spark.table("samples.wanderbricks.destinations")

## 2.2 · La tabla resultante

In [0]:
ingresos_destinos = (
    bookings.alias("b")
    .join(
        properties.alias("p"),
        F.col("b.property_id") == F.col("p.property_id"),
        "inner"
    )
    .join(
        destinations.alias("d"),
        F.col("p.destination_id") == F.col("d.destination_id"),
        "inner"
    )
    .groupBy(
        F.col("d.destination").alias("destino"),
        F.col("d.country").alias("pais")
    )
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(F.sum("b.total_amount"), 2).alias("ingresos_totales"),
        F.round(F.avg("b.total_amount"), 2).alias("promedio_reserva")
    )
    .orderBy(F.desc("ingresos_totales"))
)

display(ingresos_destinos)

## 2.3 · El recorrido del dato

*Cuántos registros hay en cada capa y a qué corresponde cada diferencia. Si algo se
descartó, digan por qué.*

In [0]:
ingresos_destinos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bigdata_grupo35.oro.ingresos_por_destino")

print("✅ Tabla Oro creada correctamente")

In [0]:
display(
    spark.table("bigdata_grupo35.oro.ingresos_por_destino")
)

**Interpretación:** *¿a qué corresponde cada diferencia entre capas?*

# 3 · Medición inicial · 15 puntos

## 3.1 · La operación que vamos a optimizar

*Qué consulta eligieron y por qué creen que es costosa. Debe ser un cruce entre dos tablas
que tarde algo medible: si responde en milisegundos, la mejora no se va a distinguir del ruido.*

In [0]:
def medir(descripcion, funcion, repeticiones=3):
    """Calienta, repite y reporta la mediana. Las tres mediciones quedan visibles."""
    
    funcion()  # calentamiento, se descarta
    
    tiempos = []
    
    for i in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempo = time.time() - inicio
        tiempos.append(tiempo)
        
        print(f"{descripcion} - ejecución {i+1}: {tiempo:.2f} segundos")
    
    mediana = statistics.median(tiempos)
    
    print(f"Mediana {descripcion}: {mediana:.2f} segundos")
    
    return mediana

In [0]:
# Consulta original, sin optimización

bookings = spark.table("bigdata_grupo35.plata.bookings")
properties = spark.table("samples.wanderbricks.properties")

consulta_original = (
    bookings.alias("b")
    .join(
        properties.alias("p"),
        F.col("b.property_id") == F.col("p.property_id"),
        "inner"
    )
    .groupBy(F.col("p.property_type"))
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(F.sum("b.total_amount"), 2).alias("ingresos_totales")
    )
)

print("✅ Consulta original creada")

In [0]:
tiempo_antes = medir(
    "ANTES - consulta original",
    lambda: consulta_original.collect()
)

## 3.2 · El plan de ejecución inicial

*Ejecuten `.explain()` y dejen la salida visible. Después señalen dónde está el costo.*

In [0]:
from pyspark.sql.functions import broadcast

bookings = spark.table("bigdata_grupo35.plata.bookings")
properties = spark.table("samples.wanderbricks.properties")

consulta_optimizada = (
    bookings.alias("b")
    .join(
        broadcast(properties.alias("p")),
        F.col("b.property_id") == F.col("p.property_id"),
        "inner"
    )
    .groupBy(F.col("p.property_type"))
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(F.sum("b.total_amount"), 2).alias("ingresos_totales")
    )
)

print("✅ Consulta optimizada creada")

**Qué vemos en el plan:**

*¿Qué estrategia eligió Spark? ¿Dónde aparece un `Exchange`? ¿Qué se está moviendo entre
máquinas y por qué eso es caro?*

## 3.3 · Los tres tiempos

In [0]:
def medir(descripcion, funcion, repeticiones=3):
    """Calienta, repite y reporta la mediana."""
    
    # Calentamiento: se ejecuta pero no se cuenta
    funcion()
    
    tiempos = []
    
    for _ in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempos.append(time.time() - inicio)

    mediana = statistics.median(tiempos)

    print(f"{descripcion}")
    print(f"   mediciones : {', '.join(f'{t:.2f}s' for t in tiempos)}")
    print(f"   MEDIANA    : {mediana:.2f}s")
    print(f"   variación  : {max(tiempos)-min(tiempos):.2f}s")
    
    return mediana


# 4 · Optimización · 20 puntos

## 4.1 · Qué técnica aplicamos y por qué

*Una sola técnica, bien elegida. Expliquen por qué esa y no otra, en función de lo que
vieron en el plan inicial.*

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
import time
import statistics

bookings = spark.table("bigdata_grupo35.plata.bookings")
properties = spark.table("samples.wanderbricks.properties")

print("✅ bookings y properties cargados")

**
4.2 · El plan de ejecución resultante**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

bookings = spark.table("bigdata_grupo35.plata.bookings")
properties = spark.table("samples.wanderbricks.properties")

consulta_optimizada = (
    bookings.alias("b")
    .join(
        broadcast(properties.alias("p")),
        F.col("b.property_id") == F.col("p.property_id"),
        "inner"
    )
    .groupBy(F.col("p.property_type"))
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(F.sum("b.total_amount"), 2).alias("ingresos_totales")
    )
)

print("✅ consulta_optimizada creada correctamente")

In [0]:
consulta_optimizada.explain()

In [0]:
import time
import statistics

def medir(descripcion, funcion, repeticiones=3):
    # Calentamiento: se ejecuta una vez y no se cuenta
    funcion()

    tiempos = []

    for _ in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempos.append(time.time() - inicio)

    mediana = statistics.median(tiempos)

    print(descripcion)
    print("   mediciones :", ", ".join(f"{t:.2f}s" for t in tiempos))
    print(f"   MEDIANA    : {mediana:.2f}s")
    print(f"   variación  : {max(tiempos)-min(tiempos):.2f}s\n")

    return mediana

print("✅ Función medir creada correctamente")

In [0]:
from pyspark.sql import functions as F

bookings = spark.table("bigdata_grupo35.plata.bookings")
properties = spark.table("samples.wanderbricks.properties")

# Desactivar broadcast para forzar el plan original

consulta_original = (
    bookings.alias("b")
    .join(
        properties.alias("p"),
        F.col("b.property_id") == F.col("p.property_id"),
        "inner"
    )
    .groupBy(F.col("p.property_type"))
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(F.sum("b.total_amount"), 2).alias("ingresos_totales")
    )
)

print("✅ consulta_original creada")

In [0]:
consulta_original.explain()

In [0]:
tiempo_antes = medir(
    "ANTES · SortMergeJoin",
    lambda: consulta_original.collect()
)

In [0]:
tiempo_despues = medir(
    "DESPUÉS - BroadcastHashJoin",
    lambda: consulta_optimizada.collect()
)

In [0]:
mejora = (tiempo_antes - tiempo_despues) / tiempo_antes * 100

print("═" * 50)
print(f"{'SortMergeJoin':<30}{tiempo_antes:>8.2f}s")
print(f"{'BroadcastHashJoin':<30}{tiempo_despues:>8.2f}s")
print("─" * 50)
print(f"{'Mejora':<30}{mejora:>8.1f}%")
print("═" * 50)

## 4.3 · Qué cambió entre los dos planes

*Comparen los dos planes de forma explícita. No basta con poner los dos números: hay que
decir qué operación desapareció o se redujo y por qué eso baja el costo.*

| | Antes | Después |
|---|--|--|
| Estrategia de cruce |SortMergeJoin | BroadcastHashJoin|
| Movimiento de datos |Exchange para repartir los datos de las tablas |BroadcastExchange para enviar la tabla pequeña |
| Ordenamiento |Se requiere ordenar los datos para el SortMergeJoin | No se necesita ese ordenamiento|

Qué cambió: La consulta original utilizaba SortMergeJoin, que necesita reorganizar los datos para realizar el cruce. Después de aplicar broadcast sobre properties, el plan cambió a BroadcastHashJoin. La tabla pequeña se envía a los ejecutores y se evita repartir de la misma manera la tabla grande. Además, al utilizar el BroadcastHashJoin no se requiere el mismo proceso de ordenamiento del SortMergeJoin. Por esto se reduce el trabajo de la operación distribuida.

## 4.4 · Los tiempos, comparados

In [0]:
tiempo_despues = medir(
    "DESPUÉS · BroadcastHashJoin",
    lambda: consulta_optimizada.collect()
)

mejora = (tiempo_antes - tiempo_despues) / tiempo_antes * 100

print("═" * 50)
print(f"{'SortMergeJoin':<30}{tiempo_antes:>8.2f}s")
print(f"{'BroadcastHashJoin':<30}{tiempo_despues:>8.2f}s")
print("─" * 50)
print(f"{'Mejora':<30}{mejora:>8.1f}%")
print("═" * 50)

**Conclusión de la optimización:**
### Conclusión de la optimización

La consulta original con SortMergeJoin tuvo una mediana de **1.17 segundos**.
Después de aplicar BroadcastHashJoin, la mediana fue de **0.98 segundos**.

Esto representa una mejora del **16.5 %** en el tiempo de ejecución.

La mejora se relaciona con el cambio en el plan de ejecución. Con BroadcastHashJoin,
se utiliza la tabla más pequeña mediante broadcast, reduciendo el movimiento de datos
necesario para realizar el cruce. Por esta razón, la consulta necesitó menos trabajo
distribuido y presentó un menor tiempo de ejecución.

# 5 · Límites de la optimización · 10 puntos

*¿En qué condiciones esta misma técnica dejaría de servir, o incluso empeoraría el resultado?*

La técnica BroadcastHashJoin funciona bien cuando una de las tablas que participa
en el cruce es relativamente pequeña. En nuestro caso utilizamos properties como
la tabla que se envía a los ejecutores.

Esta optimización podría dejar de ser conveniente si la tabla properties aumentara
mucho de tamaño. En ese caso, enviar una copia de la tabla a cada ejecutor consumiría
más memoria y aumentaría el tráfico de datos, por lo que la consulta podría tardar
más o incluso presentar problemas de memoria.

También podría dejar de ser útil si el volumen de bookings fuera mucho menor.
Cuando las tablas son pequeñas, la diferencia entre las estrategias puede ser muy
pequeña frente a la variación normal de los tiempos de ejecución.

Por lo tanto, en Wanderbricks el BroadcastHashJoin tiene sentido mientras
properties pueda considerarse una tabla pequeña en comparación con bookings.
Si el tamaño de properties creciera considerablemente, sería necesario revisar
el plan de ejecución y comparar nuevamente las estrategias.

# 6 · Cierre del caso · 10 puntos

## 6.1 · La pregunta orientadora

> Una empresa recibe datos de su portal web, de Facebook, de Instagram, de TikTok y de
> WhatsApp Business. **¿Bajo qué paradigma, con qué herramientas y con qué arquitectura
> debería analizarlos?**

*Un par de párrafos, no un ensayo. La respuesta debe apoyarse en lo que construyeron
durante el curso, no en generalidades.*


Para poder analizar los datos provenientes del portal web, Facebook, Instagram, TikTok y
WhatsApp Business utilizaríamos un **paradigma Lakehouse**, porque permite trabajar
con diferentes tipos de datos y organizarlos en capas. En nuestro proyecto
Wanderbricks utilizamos esta idea mediante las capas **Bronce, Plata y Oro**, donde
los datos pueden conservarse, limpiarse y transformarse hasta llegar a información
lista para responder preguntas de negocio.

Las herramientas serían almacenamiento en formato Delta y procesamiento distribuido
con Spark/Databricks, junto con mecanismos de gobierno como catálogos, esquemas,
permisos y linaje. La arquitectura tendría las fuentes de datos en la entrada, una
capa Bronce para conservar los datos originales, una capa Plata para limpiarlos y
transformarlos y una capa Oro para generar resultados agregados para los usuarios.
Esta estructura permite integrar las diferentes fuentes y posteriormente consumir
la información para análisis y toma de decisiones.

## 6.2 · Conclusiones del proyecto

*Qué funcionó, qué no, y qué harían distinto si empezaran de nuevo. Deben derivarse de sus
resultados, no de impresiones generales sobre la herramienta.*

### Qué funcionó

La organización de los datos en las capas Bronce, Plata y Oro permitió llevar el
caso de Wanderbricks desde los datos originales hasta una tabla agregada orientada
a una pregunta de negocio. También fue posible identificar el linaje entre las
capas y aplicar permisos diferenciados.

En la parte de optimización, el cambio de SortMergeJoin a BroadcastHashJoin
produjo una mejora medible. La mediana pasó de **1.17 segundos** a **0.98 segundos**,
lo que representa una mejora del **16.5 %**. Además, el cambio se pudo comprobar
directamente en los planes de ejecución.

### Qué no funcionó

Durante el desarrollo tuvimos algunos problemas relacionados con las variables y
la configuración de Spark. Por ejemplo, la configuración
spark.sql.autoBroadcastJoinThreshold no estaba disponible en nuestro entorno,
por lo que tuvimos que cambiar la forma de construir la consulta original para
poder comparar las estrategias.

### Qué haríamos distinto

Si empezáramos nuevamente, organizaríamos desde el comienzo las celdas de medición
y las dependencias entre variables para evitar tener que reconstruir consultas.
También revisaríamos desde el inicio las características del entorno de Databricks
antes de utilizar configuraciones específicas.

Además, mantendríamos la revisión del plan de ejecución antes y después de cualquier
optimización, porque el resultado de tiempo por sí solo no permite explicar por qué
una consulta mejoró.

# 7 · Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| *( Paula Andrea Celis Cano )* |Desarrollo de la EA3, construcción de la capa Oro, medición y optimización de la consulta  | Explicar la capa Oro, los planes de ejecución, la comparación de tiempos y los límites de la optimización|


**Uso de asistentes de IA:**

Se utilizó un asistente de IA como apoyo para comprender algunos conceptos,
revisar la sintaxis del código y organizar la estructura del notebook. 


# ⭐ Opcional · hasta 5 puntos adicionales

*Elijan **una** de las dos opciones. Es opcional: no hacerlo no resta nada.*

### Opción A · Un tablero sobre la capa oro

*Tres visualizaciones que respondan la pregunta de negocio de su capa oro. Peguen la captura
aquí.*

### Opción B · Un espacio de Genie sobre la capa oro

*Tres preguntas en lenguaje natural con sus respuestas, y **qué tuvieron que ajustar** para
que Genie entendiera: nombres de columnas, descripciones, instrucciones del espacio.*

*Eso último es lo más interesante del ejercicio: si Genie no entiende su tabla, es porque la
tabla no está bien diseñada.*

---
# ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] La capa oro existe y sus columnas se entienden sin explicación
- [ ] Las **tres mediciones** están visibles, antes y después, con la mediana
- [ ] Los **dos planes de ejecución** están en el notebook, con la salida visible
- [ ] Está escrito por qué mejoró y cuándo la técnica sería contraproducente
- [ ] La pregunta orientadora está respondida, apoyada en lo que construyeron
- [ ] No quedaron textos en cursiva de esta plantilla sin reemplazar
- [ ] La sección 7 coincide con lo que cada uno sustenta en el video
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] El `.ipynb` está confirmado en el repositorio, en `/ea3`
- [ ] El HTML con salidas visibles está subido a Canvas

### ⚠️ Sobre la cuota

Este proyecto pide ejecutar la misma consulta **ocho veces** —calentamiento y tres
mediciones, dos veces—. Eso gasta bastante más de lo habitual.
**No dejen las mediciones para la última noche.**

---

**Entrega: domingo 4 de octubre, 11:59 p. m.**